# Treino SFT — tool-use calibrado + reasoning (QLoRA + Unsloth)

**Branch `phronesis-thinking`.** GPU alvo: **L4** (Colab Pro).

Ordem obrigatória:
1. Rodar com `USE_GUINEA_PIG = True` (Qwen3-1.7B) para validar o pipeline (loss desce, salva, carrega).
2. Só então treinar o alvo deste experimento, `MODEL_SIZE = "8b"`.
   **Antes, estimar créditos e reportar ao usuário; > 15 créditos → parar e discutir.**

## Mudanças de treino nesta rodada (962 exemplos)

A avaliação pós-treino (`data/eval/thinking_8b_eval_notes.md`) mostrou que lotes
corretivos pequenos **não generalizavam** — o bug de divisão inteira reapareceu 3x em
contextos novos, a fabricação de fonte vazou pra cenários não treinados. Diagnóstico:
com batch efetivo 16 e 1 época, o dataset inteiro dava **~57 passos de otimização** —
regime de sub-treino, não de overfit.

| | antes | agora |
|---|---|---|
| épocas | 1 (~57 passos) | **2 (~114 passos)** |
| LoRA r / alpha | 16 / 32 | **32 / 64** |
| weight_decay | 0.0 | **0.01** |
| warmup_ratio | 0.03 | **0.05** |
| NEFTune | — | **alpha 5** (se a versão suportar) |
| validação | **nenhuma** | **5% estratificado por camada** |

O split de validação é o que torna isso mensurável: treinar 2+ épocas sem val loss é
às cegas — não dá pra distinguir "a época extra ajudou" de "começou a decorar". A
célula de treino imprime a curva e avisa se a val loss virar pra cima.

Antes de rodar: a célula abaixo dá `git clone`/`git pull` do repo `Conatus-Phronesis` usando o secret
`GH_TOKEN` já configurado no Colab (ícone de chave 🔑 na barra lateral) — não precisa de upload manual nem Drive.
Este notebook também serve para a **Fase 0** (baseline): pule para a última seção sem treinar.

In [ ]:
# Puxa (ou atualiza) o repositório com data/clean/train.jsonl, configs/ e src/,
# garante o branch phronesis-thinking (dataset com <think> — o main não tem isso),
# e autentica no Hugging Face (evita rate limit anônimo / permite modelos privados).
# GH_TOKEN e HF_TOKEN vêm do Colab Secrets (ícone de chave na sidebar).
import os
from google.colab import userdata
from huggingface_hub import login

GH_TOKEN = userdata.get("GH_TOKEN")
GH_USER = "devlucascfarias"
REPO_NAME = "Conatus-Phronesis"
BRANCH = "phronesis-thinking"
REPO = f"/content/{REPO_NAME}"
REPO_URL = f"https://{GH_TOKEN}@github.com/{GH_USER}/{REPO_NAME}.git"

if os.path.isdir(f"{REPO}/.git"):
    !cd {REPO} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO}

!cd {REPO} && git branch --show-current

del GH_TOKEN, REPO_URL  # não deixa o token solto na sessão do notebook além do necessário

login(token=userdata.get("HF_TOKEN"))

In [ ]:
%pip install -q unsloth
%pip install -q --no-deps trl peft accelerate bitsandbytes
import torch
print(torch.cuda.get_device_name(0))

In [ ]:
# ---- Configuração (espelha configs/train_config.yaml) ----
USE_GUINEA_PIG = True   # PRIMEIRO True (1.7B, barato); depois False para o alvo 8B
MODEL_SIZE = "8b"

_MODEL_BY_SIZE = {
    "4b": "Qwen/Qwen3-4B-Instruct-2507",  # referência sem thinking
    "8b": "Qwen/Qwen3-8B",                # alvo híbrido deste branch
}

# REPO já foi definido na célula de clone/pull acima (/content/Conatus-Phronesis)
MODEL = "Qwen/Qwen3-1.7B" if USE_GUINEA_PIG else _MODEL_BY_SIZE[MODEL_SIZE]
TAG = "1p7b" if USE_GUINEA_PIG else MODEL_SIZE
MAX_SEQ_LEN = 4096
TRAIN_FILE = f"{REPO}/data/clean/train.jsonl"

## Retomando uma sessão (adapter já treinado, sem retreinar)

Se você já treinou nesta VM e só reiniciou a sessão Python (`Restart session` — o
disco em `/content` continua, só as variáveis foram perdidas), **pule as células
4–10** (elas montam `train.jsonl` e treinam do zero) e rode direto a célula abaixo,
que carrega o `outputs/adapter_{TAG}` já salvo. Depois siga normalmente a partir da
célula de sanidade (Fase 0/eval/demo).

Se o runtime foi desconectado/resetado de verdade (VM nova, `/content` vazio), o
adapter treinado foi perdido — aí sim precisa rodar 4–10 de novo (retreinar).

In [ ]:
# Carrega o adapter já treinado (pule as células 4-10 se for rodar esta).
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=f"{REPO}/outputs/adapter_{TAG}",  # aponta pro adapter salvo, nao pro base
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print(f"Adapter outputs/adapter_{TAG} carregado, pronto pra inferencia.")

In [ ]:
# Renderiza data/clean/dataset.jsonl no chat template do Qwen3 + máscara de loss (seção 4.3),
# gerando o train.jsonl que a célula seguinte consome. Os blocos <think> dos itens difíceis
# ficam dentro do span treinável do assistant; turnos sem reasoning continuam normais.
!cd {REPO} && python src/build_dataset.py data/clean/dataset.jsonl --out data/clean/train.jsonl --model {MODEL} --max-len {MAX_SEQ_LEN}

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
# r=32/alpha=64 (era 16/32, herdado do 4B com 579 exemplos): o dataset cresceu 66% e
# ganhou comportamentos novos (thinking calibrado por camada, desistência graciosa,
# verificação-antes-de-fechar). Ver configs/train_config.yaml para o racional completo.
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from collections import Counter, defaultdict
import random

from datasets import load_dataset

# train.jsonl vem do build_dataset.py e já contém o campo 'text' renderizado
# via tokenizer.apply_chat_template (nunca concatenar strings na mão).
full = load_dataset("json", data_files=TRAIN_FILE, split="train")

# Split de validação ESTRATIFICADO por camada. Antes não existia validação nenhuma —
# com 1 época isso passava, mas treinar 2+ épocas sem val loss é às cegas: não dá pra
# distinguir "a época extra ajudou" de "começou a decorar". Estratificado porque a
# camada 2 é ~40% do dataset; um split aleatório sub-representaria as outras camadas.
VAL_SPLIT = 0.05
by_layer = defaultdict(list)
for i, layer in enumerate(full["layer"]):
    by_layer[layer].append(i)

rng = random.Random(42)
val_idx = []
for layer, idxs in sorted(by_layer.items()):
    idxs = idxs[:]           # nao embaralha a lista original
    rng.shuffle(idxs)
    k = max(1, round(len(idxs) * VAL_SPLIT))
    val_idx.extend(idxs[:k])

val_set = set(val_idx)
train_idx = [i for i in range(len(full)) if i not in val_set]

keep_only_text = lambda ds: ds.remove_columns([c for c in ds.column_names if c != "text"])
dataset = keep_only_text(full.select(train_idx))
eval_dataset = keep_only_text(full.select(sorted(val_idx)))

print(f"treino: {len(dataset)} | validação: {len(eval_dataset)}")
print("val por camada:", dict(sorted(Counter(full.select(sorted(val_idx))["layer"]).items())))
print("\n", dataset[0]["text"][:600])

In [ ]:
import inspect

from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

# Alguns parâmetros mudaram de nome/existência entre versões do TRL/transformers
# (`evaluation_strategy` virou `eval_strategy` na 4.41; `neftune_noise_alpha` só existe
# da 4.35 em diante). Detecta em runtime em vez de assumir a versão do Colab.
_sft_params = set(inspect.signature(SFTConfig.__init__).parameters)
extra = {}

if "eval_strategy" in _sft_params:
    extra["eval_strategy"] = "steps"
elif "evaluation_strategy" in _sft_params:
    extra["evaluation_strategy"] = "steps"
if extra:                                   # só faz sentido se o eval foi aceito
    extra["eval_steps"] = 15
    extra["per_device_eval_batch_size"] = 2

# NEFTune: ruído nos embeddings durante o treino, técnica específica pra generalização
# em instruction tuning — é exatamente o problema medido (lotes corretivos não
# generalizavam pra contextos novos, ver data/eval/thinking_8b_eval_notes.md).
if "neftune_noise_alpha" in _sft_params:
    extra["neftune_noise_alpha"] = 5

print("extras aceitos por esta versão:", sorted(extra))

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=eval_dataset if extra.get("eval_strategy") or extra.get("evaluation_strategy") else None,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # batch efetivo ~16
        num_train_epochs=2,              # era 1: ~60 passos era regime de sub-treino
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,               # era 0.03 (dava só ~4 passos de warmup)
        weight_decay=0.01,               # era 0.0 (default do HF)
        logging_steps=5,
        seed=42,
        output_dir=f"outputs_{TAG}",
        report_to="none",
        **extra,
    ),
)

# Alinhado à máscara da seção 4.3: loss só nos turnos do assistant;
# system/user/<tool_response> (renderizado dentro de um turno user pelo template) ficam fora.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

steps = (len(dataset) // 16) * 2
print(f"~{steps} passos de otimização no total (antes: ~{len(dataset)//16} com 1 época)")

In [ ]:
# Verificação da máscara antes de treinar (equivalente ao --show-masks do build_dataset.py)
sample = trainer.train_dataset[0]
ids, labels = sample["input_ids"], sample["labels"]
trainable = [t for t, l in zip(ids, labels) if l != -100]
print("TREINÁVEL:", tokenizer.decode(trainable)[:500])
assert any(l != -100 for l in labels), "máscara zerou tudo — conferir template"

In [ ]:
stats = trainer.train()
print(stats)

# Curva de loss — é isto que decide se 2 épocas foi o ponto certo. Leitura:
#  - val loss ainda caindo no fim  → ainda sub-treinado, a próxima rodada pode ir a 3 épocas
#  - val loss virou pra cima (train continua caindo) → começou a decorar, volta pra 1-2
#  - val loss estável no fim       → 2 épocas está no ponto
hist = trainer.state.log_history
train_pts = [(h.get("epoch"), h["loss"]) for h in hist if "loss" in h]
eval_pts = [(h.get("epoch"), h["eval_loss"]) for h in hist if "eval_loss" in h]

print("\n época  train_loss   val_loss")
for ep, tl in train_pts:
    closest = min(eval_pts, key=lambda e: abs((e[0] or 0) - (ep or 0)), default=None)
    vl = f"{closest[1]:.4f}" if closest and abs((closest[0] or 0) - (ep or 0)) < 0.06 else ""
    print(f"  {ep:>5.2f}   {tl:>8.4f}   {vl:>8}")

if eval_pts:
    print(f"\nval loss: {eval_pts[0][1]:.4f} (início) → {eval_pts[-1][1]:.4f} (fim)")
    if eval_pts[-1][1] > min(v for _, v in eval_pts) * 1.02:
        best = min(eval_pts, key=lambda e: e[1])
        print(f"ATENÇÃO: val loss subiu depois do mínimo ({best[1]:.4f} na época {best[0]:.2f}) "
              f"— sinal de overfit, considere reduzir épocas na próxima rodada.")
else:
    print("\n(sem val loss: esta versão do TRL não aceitou os parâmetros de eval)")

In [ ]:
# Salvar adapter + merged 16-bit (para a Fase 4)
model.save_pretrained(f"{REPO}/outputs/adapter_{TAG}")
tokenizer.save_pretrained(f"{REPO}/outputs/adapter_{TAG}")
model.save_pretrained_merged(f"{REPO}/outputs/merged_{TAG}", tokenizer, save_method="merged_16bit")
print("Salvo. Teste de sanidade de geração abaixo.")

In [ ]:
# Sanidade: checkpoint carrega e gera no template correto
import json
FastLanguageModel.for_inference(model)
tools = json.load(open(f"{REPO}/configs/tools.json", encoding="utf-8"))["tools"]
msgs = [{"role": "user", "content": "Quanto tá o dólar hoje?"}]
prompt = tokenizer.apply_chat_template(msgs, tools=tools, tokenize=False, add_generation_prompt=True, enable_thinking=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
# repetition_penalty: greedy puro entra em loop de repeticao variando um numero a cada linha
# (evade no_repeat_ngram_size, que so bloqueia n-gramas identicos) — visto no eval do held-out.
out = model.generate(**inputs, max_new_tokens=2048, do_sample=False,
                     repetition_penalty=1.15, no_repeat_ngram_size=8,
                     pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Avaliação (Fase 0 baseline / Fase 4 treinado)

Fase 0 (sem treino): rode a célula abaixo direto, sem executar as células de treino.
Fase 4: aponte `--adapter` para o adapter salvo.

Ao final da sessão, **anote os créditos consumidos** e atualize a tabela da seção 6 do PLANO.md.

In [ ]:
# --max-new-tokens 512: o testset.jsonl não tem camada 3 (só 0/0.5/1/2/C), e a métrica só
# lê o primeiro <tool_call> e o preâmbulo antes dele — com os tetos de <think> dessas
# camadas (15-40 palavras), uma resposta típica tem ~170 tokens. O default de 2048 só é
# gasto quando o modelo não para sozinho (divagação/loop), custando até 4x mais tempo sem
# mudar nenhuma métrica. O eval de rigor matemático (camada 2/3) segue com 2048.

# Fase 0 — baseline do modelo cru:
!cd {REPO} && python src/eval_harness.py --model {MODEL} --max-new-tokens 512 --out outputs/baseline_metrics.json

# Fase 4 — modelo treinado (adapter descomentado por padrao; ja causou confusao uma vez rodar sem):
!cd {REPO} && python src/eval_harness.py --model {MODEL} --adapter outputs/adapter_{TAG} --max-new-tokens 512 --out outputs/trained_metrics.json

## Rigor matemático — held-out de camada 2/3 (40 itens, nunca usados em treino)

`eval_harness.py` acima só mede escolha de tool e comprimento de preâmbulo — nunca
mediu se o raciocínio em si está correto. `data/eval/math_rigor_testset.jsonl` tem
40 itens com resolução de referência (`\boxed{}` quando aplicável) reservados
exatamente pra isso.

A célula abaixo só **gera** as respostas do modelo (roda aqui, GPU). A nota é dada
depois, offline, por Claude Code lendo par a par (`src/score_math_rigor.py prepare`
→ Claude Code aplica `prompts/judge_math_rigor.md` → `src/score_math_rigor.py
apply`) — comparação simbólica de LaTeX (vetores, matrizes, provas sem resultado
numérico fechado) é frágil demais pra um comparador mecânico, mesmo padrão já usado
pra curadoria do dataset em `src/judge_data.py`.

In [ ]:
# Gera as respostas do modelo treinado pro held-out de rigor matemático.
# Baixe outputs/math_rigor_responses.jsonl depois (Files, barra lateral) e traga
# pra sessão local — o julgamento (Claude Code) roda offline, fora do Colab.
!cd {REPO} && python src/run_math_rigor_eval.py --model {MODEL} --adapter outputs/adapter_{TAG} --out outputs/math_rigor_responses.jsonl

## Fase 5 — demo do agente com busca REAL na internet

Roda o **8B treinado** (em memória, na mesma sessão) num loop de agente de verdade: gera → detecta `<tool_call>` → **executa** → devolve o resultado → gera de novo.

- `web_search` usa o **Ollama web search** (secret `OLLAMA_SEARCH_KEY` do Colab); cai para DuckDuckGo se falhar.
- `python_sandbox` executa o código de verdade (subprocess isolado, timeout 5s, imports na whitelist).

Testa os 3 fluxos: pergunta volátil (busca), cálculo (sandbox) e conversa (sem tool).

In [ ]:
%pip install -q ddgs   # fallback de busca; provedor principal e o Ollama (OLLAMA_SEARCH_KEY)

import os, sys, json
from google.colab import userdata
sys.path.insert(0, f"{REPO}/src")

# chave de busca do Ollama (secret do Colab) -> ambiente, pra o executor enxergar
os.environ["OLLAMA_SEARCH_KEY"] = userdata.get("OLLAMA_SEARCH_KEY")

from inference_loop import run_agent
tools = json.load(open(f"{REPO}/configs/tools.json", encoding="utf-8"))["tools"]

FastLanguageModel.for_inference(model)   # usa o modelo treinado que ja esta em memoria

def generate(messages):
    prompt = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    # do_sample=True com os parametros recomendados do Qwen3-8B (generation_config.json real do
    # modelo) — greedy puro entra em loop de repeticao variando um numero a cada linha, medido em
    # data/eval/thinking_8b_eval_notes.md. Sem seed fixa aqui de proposito (uso interativo/demo).
    out = model.generate(**inputs, max_new_tokens=2048, do_sample=True,
                         temperature=0.6, top_k=20, top_p=0.95,
                         no_repeat_ngram_size=8, repetition_penalty=1.15,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for pergunta in [
    "Quanto ta o dolar hoje?",          # volatil -> web_search real
    "Quanto e 37,5% de 18.420?",        # calculo -> python_sandbox
    "Me conta uma curiosidade aleatoria.",  # conversa -> sem tool
]:
    print("="*70)
    print(f"voce> {pergunta}\n")
    run_agent([{"role": "user", "content": pergunta}], generate, verbose=True)

In [ ]:
# Bateria de avaliação manual — reusa generate()/tools/run_agent já definidos na célula acima.
# Cobre: os 3 casos que falharam ao vivo antes do batch_022 (conferir se o corretivo pegou)
# + uma amostra de outras camadas pra visão geral de qualidade.

perguntas_avaliacao = [
    # --- regressões documentadas, checando se batch_022 corrigiu ---
    "Quanto é 37,5% de 18.420?",                         # camada 2: resposta final tinha alucinado 18.422/18.423
    "Quanto tá o dólar hoje?",                            # camada 1: tinha inventado "Banco Central" além da fonte real
    "Me conta uma curiosidade sobre polígonos regulares.", # camada C: ângulo de 1000 lados saiu errado (179,999... em vez de 179,64)
    "Calcule 62,5% de 4.960.",                            # camada 2: sem contexto de dinheiro — checa se ainda inventa "reais"
    "Quem venceu o jogo do Brasil ontem?",                # camada 1: checa se cita só fontes reais quando há mais de uma

    # --- amostra geral de outras camadas ---
    "Qual é o segundo maior planeta do sistema solar?",   # camada 0: deve responder seco, sem tool
    "Quem era o presidente do Brasil na Copa de 94?",     # camada 0.5: parece atual, mas é histórico
    "Discussão séria: pizza com abacaxi é crime ou aceitável?",  # camada C: conversa, sem tool
    "Resolva \\(2x^2-11x+12=0\\) por Bhaskara.",          # camada 2: matemática de rotina, deve chamar sandbox
]

for pergunta in perguntas_avaliacao:
    print("=" * 70)
    print(f"voce> {pergunta}\n")
    run_agent([{"role": "user", "content": pergunta}], generate, verbose=True)

In [ ]:
# Mesma bateria, agora com sampling nos valores que o Qwen3-8B recomenda por padrão
# (confirmado direto no generation_config.json real do modelo, modo thinking):
# temperature=0.6, top_k=20, top_p=0.95, do_sample=true.
# Objetivo: comparar contra a rodada greedy (do_sample=False) acima — sampling costuma
# escapar de loops de repetição que o greedy puro trava (autovalores, EDO, campo elétrico,
# incerteza quântica no held-out).

def generate_sampling(messages):
    prompt = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=2048, do_sample=True,
                         temperature=0.6, top_k=20, top_p=0.95,
                         no_repeat_ngram_size=8, repetition_penalty=1.15,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for pergunta in perguntas_avaliacao:
    print("=" * 70)
    print(f"voce> {pergunta}\n")
    run_agent([{"role": "user", "content": pergunta}], generate_sampling, verbose=True)